# Article Figures Generator
**Purpose:** Read existing CSV results and generate publication-ready PDF figures.

**Requirements:**
- Color-blind friendly palette (Wong's)
- Bold axis labels
- PDF output format
- Line charts (no bar charts)

**Usage:** Run all cells. PDFs are saved to the same folder as this notebook.


## Data sources (where the figure data comes from)

All figures are generated **from CSV/Excel files** produced by other scripts. Knowing the source helps interpret and reproduce the results.

| Figure group | Data files | Generated by |
|--------------|------------|--------------|
| **Ablation (SO/ACS)** | `ablation_results/{so,acs}_ablation_summary.csv` | **`algorithms/ablation_study.py`** — also writes `{dataset}_ablation_raw_results.xlsx`. Two experiments: (1) varying ε with fixed δ; (2) varying δ with fixed ε. Compares FPGrowth (BruteForce) vs RW_Direct (RW). |
| **Scalability** | `graphs/{acs,so}_scalability_attributes.csv`, `graphs/{acs,so}_scalability_rows.csv` | **`algorithms/all_subgroups_loop.py`** (mode 2 = rows, mode 3 = attributes) — appends runs to these CSVs. Same loop can write `graphs/{ds}_homogeneity_results.xlsx` (mode 0). Alternatively **`algorithms/scalability_test.py`** writes `scalability_results_{ds}.xlsx`; the CSVs in `graphs/` are the ones plotted here. |
| **Benchmark P2 (largest δ)** | `problem_2_3_algorithms/.../find_delta_benchmark_results.csv` | **`problem_2_3_algorithms/benchmark_find_delta.py`** — also writes `find_delta_benchmark_results.xlsx`. Tests runtime vs ε for "find largest delta" across rules. |
| **Benchmark P3 (smallest ε)** | `problem_2_3_algorithms/.../find_epsilon_benchmark_results.csv` | **`problem_2_3_algorithms/benchmark_find_epsilon.py`** — also writes `find_epsilon_benchmark_results.xlsx`. Tests runtime vs δ for "find smallest epsilon". ACS plot excludes Rule 10 (outliers). |

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend (Colab-safe)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Wong's color-blind safe palette ──────────────────────────────────
WONG = {
    "blue":    "#0072B2",
    "orange":  "#E69F00",
    "green":   "#009E73",
    "yellow":  "#F0E442",
    "skyblue": "#56B4E9",
    "red":     "#D55E00",
    "pink":    "#CC79A7",
    "black":   "#000000",
}
WONG_LIST = list(WONG.values())

# ── Canonical display names/colors (match scalability figures) ───────
ALGO_DISPLAY = {
    "FPGrowth": "BruteForce",
    "BruteForce": "BruteForce",
    "RW_Direct": "RW",
    "RW_Agreement": "RW",
    "RW_unlearning": "RW",
    "RW": "RW",
}
ALGO_COLOR = {
    "BruteForce": WONG["blue"],
    "RW": WONG["orange"],
}
ALGO_MARKER = {
    "BruteForce": "s",
    "RW": "o",
}

def display_algo(raw_name: str) -> str:
    return ALGO_DISPLAY.get(raw_name, raw_name)

# ── Global matplotlib style ──────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelweight": "bold",
    "axes.labelsize": 13,
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.1,
})

# ── Paths (works from notebook/terminal/any cwd) ─────────────────────
# Notebooks don't reliably define __file__. We resolve the article_figures
# directory by checking common run locations.

def _resolve_article_figures_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "article_figures",
        cwd / "homogeneity_repo" / "article_figures",
        cwd / "homogenity_project" / "article_figures",
        cwd / "project_updated" / "homogenity_project" / "article_figures",
    ]
    for c in candidates:
        if (c / "generate_figures.ipynb").exists():
            return c
    return cwd

BASE_PATH = _resolve_article_figures_dir()
BASE = str(BASE_PATH)                 # article_figures folder
PROJECT = str(BASE_PATH.parent)       # homogenity_project root

SO_ABLATION_CSV   = os.path.join(PROJECT, "ablation_results", "so_ablation_summary.csv")
ACS_ABLATION_CSV  = os.path.join(PROJECT, "ablation_results", "acs_ablation_summary.csv")
# Benchmark results - SO dataset
BENCH_P2_SO_CSV   = os.path.join(PROJECT, "problem_2_3_algorithms", "benchmark_two_phase_FIXED",
                                 "problem2_largest_delta", "find_delta_benchmark_results.csv")
BENCH_P3_SO_CSV   = os.path.join(PROJECT, "problem_2_3_algorithms", "benchmark_two_phase_FIXED",
                                 "problem3_smallest_epsilon", "find_epsilon_benchmark_results.csv")
# Benchmark results - ACS dataset
BENCH_P2_ACS_CSV  = os.path.join(PROJECT, "problem_2_3_algorithms", "benchmark_acs_full",
                                 "problem2_largest_delta", "find_delta_benchmark_results.csv")
BENCH_P3_ACS_CSV  = os.path.join(PROJECT, "problem_2_3_algorithms", "benchmark_acs_full",
                                 "problem3_smallest_epsilon", "find_epsilon_benchmark_results.csv")
# Scalability results (graphs generated from these CSVs in the section below)
GRAPHS_DIR = os.path.join(PROJECT, "graphs")

OUTPUT_DIR = BASE  # PDFs saved next to this notebook

print("✅ Setup complete")
print(f"   SO ablation CSV:  {os.path.exists(SO_ABLATION_CSV)}")
print(f"   ACS ablation CSV: {os.path.exists(ACS_ABLATION_CSV)}")
print(f"   Benchmark P2 SO CSV:  {os.path.exists(BENCH_P2_SO_CSV)}")
print(f"   Benchmark P3 SO CSV:  {os.path.exists(BENCH_P3_SO_CSV)}")
print(f"   Benchmark P2 ACS CSV: {os.path.exists(BENCH_P2_ACS_CSV)}")
print(f"   Benchmark P3 ACS CSV: {os.path.exists(BENCH_P3_ACS_CSV)}")
print(f"   Scalability attrs (acs): {os.path.exists(os.path.join(GRAPHS_DIR, 'acs_scalability_attributes.csv'))}")
print(f"   Scalability attrs (so):  {os.path.exists(os.path.join(GRAPHS_DIR, 'so_scalability_attributes.csv'))}")
print(f"   Scalability rows (acs): {os.path.exists(os.path.join(GRAPHS_DIR, 'acs_scalability_rows.csv'))}")
print(f"   Scalability rows (so):  {os.path.exists(os.path.join(GRAPHS_DIR, 'so_scalability_rows.csv'))}")


✅ Setup complete
   SO ablation CSV:  True
   ACS ablation CSV: True
   Benchmark P2 SO CSV:  True
   Benchmark P3 SO CSV:  True
   Benchmark P2 ACS CSV: True
   Benchmark P3 ACS CSV: True


In [2]:
# ── Helper: save figure as PDF (optional title shown on figure) ─────────
def save_pdf(fig, name, title=None):
    if title is not None:
        fig.suptitle(title, fontsize=10, y=1.02)
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, format="pdf")
    plt.close(fig)
    print(f"  ✅ Saved: {name}")


# ── Helper: make a short readable rule label from dict-style strings ─
def _short_rule_label(condition_str, treatment_str):
    """Turn \"{'Gender': 'Male'}\" -> \"Gender=Male\" etc."""
    import ast
    try:
        cond = ast.literal_eval(condition_str)
        treat = ast.literal_eval(treatment_str)
    except Exception:
        return f"{condition_str} → {treatment_str}"
    cond_parts = [f"{k}={v}" for k, v in cond.items()]
    treat_parts = [f"{k}={v}" for k, v in treat.items()]
    # Truncate long values
    def _trunc(s, max_len=25):
        return s if len(s) <= max_len else s[:max_len-1] + "…"
    cond_s = ", ".join(_trunc(p) for p in cond_parts)
    treat_s = ", ".join(_trunc(p) for p in treat_parts)
    return f"{cond_s} → {treat_s}"


# ── Helper: generate all 6 ablation figures for one dataset ──────────
def generate_ablation_figures(csv_path, ds_prefix, ds_label):
    """Generate 6 ablation PDFs for a given dataset."""
    df = pd.read_csv(csv_path)

    eps_data   = df[df["experiment"] == "varying_epsilon"]
    delta_data = df[df["experiment"] == "varying_delta"]

    # ── Extract series (keep raw algorithm IDs from CSV) ──────────────
    eps_agreement  = eps_data[eps_data["algorithm"] == "RW_Agreement"].sort_values("epsilon")
    delta_agreement = delta_data[delta_data["algorithm"] == "RW_Agreement"].sort_values("delta_pct")

    eps_fp  = eps_data[eps_data["algorithm"] == "FPGrowth"].sort_values("epsilon")
    eps_rw  = eps_data[eps_data["algorithm"] == "RW_Direct"].sort_values("epsilon")
    delta_fp = delta_data[delta_data["algorithm"] == "FPGrowth"].sort_values("delta_pct")
    delta_rw = delta_data[delta_data["algorithm"] == "RW_Direct"].sort_values("delta_pct")

    # For epsilon-based plots: use "K" units only when epsilons are large (SO).
    # For ACS (epsilons like 50..1000), show raw values.
    use_eps_k = (len(eps_data) > 0) and (eps_data["epsilon"].max() >= 10_000)
    eps_xlabel = "Epsilon (K)" if use_eps_k else "Epsilon"

    def _format_epsilon_tick(x, _pos=None):
        if use_eps_k:
            return f"{x/1000:.0f}"
        return f"{x:,.0f}"

    # ── 1. RW Correctness vs Epsilon ──────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(
        eps_agreement["epsilon"],
        eps_agreement["agreement_rate"],
        marker=ALGO_MARKER["RW"],
        linewidth=2.5,
        color=ALGO_COLOR["RW"],
        label="RW",
    )
    ax.fill_between(
        eps_agreement["epsilon"],
        eps_agreement["agreement_rate"],
        alpha=0.15,
        color=ALGO_COLOR["RW"],
    )
    ax.set_xlabel(eps_xlabel)
    ax.set_ylabel("Agreement with BruteForce (%)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_format_epsilon_tick))
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_correctness_varying_epsilon.pdf",
             f"Ablation ({ds_label}): RW agreement rate with BruteForce (%) as the epsilon parameter varies across experiments.")

    # ── 2. RW Correctness vs Delta ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(
        delta_agreement["delta_pct"],
        delta_agreement["agreement_rate"],
        marker=ALGO_MARKER["RW"],
        linewidth=2.5,
        color=ALGO_COLOR["RW"],
        label="RW",
    )
    ax.fill_between(
        delta_agreement["delta_pct"],
        delta_agreement["agreement_rate"],
        alpha=0.15,
        color=ALGO_COLOR["RW"],
    )
    ax.set_xlabel("Delta (% of Dataset)")
    ax.set_ylabel("Agreement with BruteForce (%)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_correctness_varying_delta.pdf",
             f"Ablation ({ds_label}): RW agreement rate with BruteForce (%) as delta (% of dataset size) varies across experiments.")

    # ── 3. Runtime vs Epsilon (LINE chart) ────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(
        eps_fp["epsilon"],
        eps_fp["runtime_seconds_mean"],
        marker=ALGO_MARKER["BruteForce"],
        linewidth=2.5,
        color=ALGO_COLOR["BruteForce"],
        label="BruteForce",
    )
    ax.plot(
        eps_rw["epsilon"],
        eps_rw["runtime_seconds_mean"],
        marker=ALGO_MARKER["RW"],
        linewidth=2.5,
        color=ALGO_COLOR["RW"],
        label="RW",
    )
    ax.set_xlabel(eps_xlabel)
    ax.set_ylabel("Average Runtime (seconds)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_format_epsilon_tick))
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_runtime_varying_epsilon.pdf",
             f"Ablation ({ds_label}): Average runtime in seconds for BruteForce and RW as the epsilon parameter varies.")

    # ── 4. Runtime vs Delta (LINE chart) ──────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(
        delta_fp["delta_pct"],
        delta_fp["runtime_seconds_mean"],
        marker=ALGO_MARKER["BruteForce"],
        linewidth=2.5,
        color=ALGO_COLOR["BruteForce"],
        label="BruteForce",
    )
    ax.plot(
        delta_rw["delta_pct"],
        delta_rw["runtime_seconds_mean"],
        marker=ALGO_MARKER["RW"],
        linewidth=2.5,
        color=ALGO_COLOR["RW"],
        label="RW",
    )
    ax.set_xlabel("Delta (% of Dataset)")
    ax.set_ylabel("Average Runtime (seconds)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_runtime_varying_delta.pdf",
             f"Ablation ({ds_label}): Average runtime in seconds for BruteForce and RW as delta (% of dataset) varies.")

    # ── 5. Param Impact — Epsilon ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    if len(eps_fp) > 0:
        ax.plot(
            eps_fp["epsilon"],
            eps_fp["homogeneity_rate"],
            marker=ALGO_MARKER["BruteForce"],
            linewidth=2.5,
            color=ALGO_COLOR["BruteForce"],
            label="BruteForce",
        )
    if len(eps_rw) > 0:
        ax.plot(
            eps_rw["epsilon"],
            eps_rw["homogeneity_rate"],
            marker=ALGO_MARKER["RW"],
            linewidth=2.5,
            color=ALGO_COLOR["RW"],
            label="RW",
        )
    ax.set_xlabel(eps_xlabel)
    ax.set_ylabel("% Rules Homogeneous")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_format_epsilon_tick))
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_homogeneity_impact_epsilon.pdf",
             f"Ablation ({ds_label}): Percentage of rules that are homogeneous for BruteForce vs RW as epsilon varies.")

    # ── 6. Param Impact — Delta ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    if len(delta_fp) > 0:
        ax.plot(
            delta_fp["delta_pct"],
            delta_fp["homogeneity_rate"],
            marker=ALGO_MARKER["BruteForce"],
            linewidth=2.5,
            color=ALGO_COLOR["BruteForce"],
            label="BruteForce",
        )
    if len(delta_rw) > 0:
        ax.plot(
            delta_rw["delta_pct"],
            delta_rw["homogeneity_rate"],
            marker=ALGO_MARKER["RW"],
            linewidth=2.5,
            color=ALGO_COLOR["RW"],
            label="RW",
        )
    ax.set_xlabel("Delta (% of Dataset)")
    ax.set_ylabel("% Rules Homogeneous")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_pdf(fig, f"ablation_{ds_prefix}_homogeneity_impact_delta.pdf",
             f"Ablation ({ds_label}): Percentage of rules that are homogeneous for BruteForce vs RW as delta varies.")

print("✅ Helper functions defined")

✅ Helper functions defined


## SO Ablation Figures (6 PDFs)

**Data source:** `ablation_results/so_ablation_summary.csv` (from **`algorithms/ablation_study.py`**; raw runs in `so_ablation_raw_results.xlsx`). Experiments: varying ε (fixed δ=10%) and varying δ (fixed ε=350k). SO epsilon values 250k–450k; delta 5%, 10%, 15%, 20%.

**What each figure shows (SO dataset):**
1. **ablation_so_correctness_varying_epsilon.pdf** — RW agreement with BruteForce (%) as ε (epsilon) varies.
2. **ablation_so_correctness_varying_delta.pdf** — RW agreement with BruteForce (%) as δ (delta, % of dataset) varies.
3. **ablation_so_runtime_varying_epsilon.pdf** — Average runtime (s) of BruteForce vs RW as ε varies.
4. **ablation_so_runtime_varying_delta.pdf** — Average runtime (s) of BruteForce vs RW as δ varies.
5. **ablation_so_homogeneity_impact_epsilon.pdf** — % of rules that are homogeneous (BruteForce vs RW) as ε varies.
6. **ablation_so_homogeneity_impact_delta.pdf** — % of rules that are homogeneous (BruteForce vs RW) as δ varies.


In [3]:
print("📊 Generating SO ablation figures...")
generate_ablation_figures(SO_ABLATION_CSV, "so", "SO Dataset")
print("Done!\n")


📊 Generating SO ablation figures...
  ✅ Saved: ablation_so_correctness_varying_epsilon.pdf
  ✅ Saved: ablation_so_correctness_varying_delta.pdf


  ✅ Saved: ablation_so_runtime_varying_epsilon.pdf


  ✅ Saved: ablation_so_runtime_varying_delta.pdf
  ✅ Saved: ablation_so_homogeneity_impact_epsilon.pdf
  ✅ Saved: ablation_so_homogeneity_impact_delta.pdf
Done!



## ACS Ablation Figures (6 PDFs)

**Data source:** `ablation_results/acs_ablation_summary.csv` (from **`algorithms/ablation_study.py`**; raw runs in `acs_ablation_raw_results.xlsx`). Same two experiments; ACS epsilon values 50–1000, fixed ε=500 for delta experiment; delta 0.01%–0.10%.

**What each figure shows (ACS dataset):**
1. **ablation_acs_correctness_varying_epsilon.pdf** — RW agreement with BruteForce (%) as ε (epsilon) varies.
2. **ablation_acs_correctness_varying_delta.pdf** — RW agreement with BruteForce (%) as δ (delta, % of dataset) varies.
3. **ablation_acs_runtime_varying_epsilon.pdf** — Average runtime (s) of BruteForce vs RW as ε varies.
4. **ablation_acs_runtime_varying_delta.pdf** — Average runtime (s) of BruteForce vs RW as δ varies.
5. **ablation_acs_homogeneity_impact_epsilon.pdf** — % of rules that are homogeneous (BruteForce vs RW) as ε varies.
6. **ablation_acs_homogeneity_impact_delta.pdf** — % of rules that are homogeneous (BruteForce vs RW) as δ varies.


In [4]:
print("📊 Generating ACS ablation figures...")
generate_ablation_figures(ACS_ABLATION_CSV, "acs", "ACS Dataset")
print("Done!\n")


📊 Generating ACS ablation figures...
  ✅ Saved: ablation_acs_correctness_varying_epsilon.pdf
  ✅ Saved: ablation_acs_correctness_varying_delta.pdf
  ✅ Saved: ablation_acs_runtime_varying_epsilon.pdf


  ✅ Saved: ablation_acs_runtime_varying_delta.pdf


  ✅ Saved: ablation_acs_homogeneity_impact_epsilon.pdf


  ✅ Saved: ablation_acs_homogeneity_impact_delta.pdf
Done!



## Scalability graphs (attributes & rows)

**Data source:** `graphs/{acs,so}_scalability_attributes.csv` and `graphs/{acs,so}_scalability_rows.csv`, produced by **`algorithms/all_subgroups_loop.py`** (mode 2 = rows, mode 3 = attributes). Rows: dataset percentage 10%–100%; attributes: number of attributes (from 3 upward). Same loop can write `graphs/{ds}_homogeneity_results.xlsx` (mode 0). A separate script **`algorithms/scalability_test.py`** writes `scalability_results_{ds}.xlsx`; the figures here use the CSVs in `graphs/`.

**What each figure shows:**
1. **acs_scalability_attributes_graph.pdf** — Average runtime (s) of BruteForce vs RW vs number of attributes (ACS).
2. **so_scalability_attributes_graph.pdf** — Average runtime (s) of BruteForce vs RW vs number of attributes (SO).
3. **acs_scalability_rows_graph.pdf** — Average runtime (s) of BruteForce vs RW vs dataset percentage (10%–100%, ACS).
4. **so_scalability_rows_graph.pdf** — Average runtime (s) of BruteForce vs RW vs dataset percentage (10%–100%, SO; log scale).

In [ ]:
# Scalability: same logic as graph_scalability_attrs.py and graph_scalability_rows.py
# Data: PROJECT/graphs/{acs,so}_scalability_attributes.csv and _scalability_rows.csv

DATASETS_SCALABILITY = ["acs", "so"]
ROW_PERCENTAGES = [x / 10 for x in range(1, 11)]
BASE_ATTRS = 3

print("📊 Generating scalability figures (from CSV in graphs/)...")

for ds in DATASETS_SCALABILITY:
    # ── 1. Attributes: runtime vs number of attributes ────────────────────
    attr_file = os.path.join(GRAPHS_DIR, f"{ds}_scalability_attributes.csv")
    if os.path.exists(attr_file):
        df_attrs = pd.read_csv(attr_file)
        df_attrs["algorithm"] = df_attrs["algorithm"].replace({"FPGrowth": "BruteForce", "RW_unlearning": "RW"})
        df_attrs = df_attrs[df_attrs["num_attributes"] >= 0].copy()
        df_attrs["actual_num_attrs"] = df_attrs["num_attributes"] + BASE_ATTRS
        df_attrs_agg = df_attrs.groupby(["algorithm", "actual_num_attrs"])["run_time_seconds"].mean().reset_index()
        x_vals = sorted(df_attrs_agg["actual_num_attrs"].unique())

        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.tick_params(axis="both", labelsize=11)
        ax.xaxis.label.set_size(13)
        ax.xaxis.label.set_weight("bold")
        ax.yaxis.label.set_size(13)
        ax.yaxis.label.set_weight("bold")
        for i, algo in enumerate(df_attrs_agg["algorithm"].unique()):
            sub = df_attrs_agg[df_attrs_agg["algorithm"] == algo].sort_values("actual_num_attrs")
            color = WONG_LIST[i % len(WONG_LIST)]
            marker = ALGO_MARKER.get(display_algo(algo), "o")
            ax.plot(sub["actual_num_attrs"], sub["run_time_seconds"], marker=marker, linewidth=2.5,
                   color=color, label=algo, markersize=8)
        ax.set_xlabel("Num of Attributes")
        ax.set_ylabel("Avg Runtime (s)")
        ax.set_xlim(left=BASE_ATTRS)
        ax.set_xticks(x_vals)
        ax.legend(loc="best", fontsize=11, frameon=True, title=None)
        ax.grid(True, alpha=0.3)
        save_pdf(fig, f"{ds}_scalability_attributes_graph.pdf",
                 f"Scalability ({ds.upper()}): Average runtime in seconds for BruteForce and RW versus the number of attributes in the dataset.")
    else:
        print(f"  ⚠ Skipping {ds} attributes: {attr_file} not found")

    # ── 2. Rows: runtime vs dataset percentage ────────────────────────────
    row_file = os.path.join(GRAPHS_DIR, f"{ds}_scalability_rows.csv")
    if os.path.exists(row_file):
        df_rows = pd.read_csv(row_file)
        df_rows["algorithm"] = df_rows["algorithm"].replace({"FPGrowth": "BruteForce", "RW_unlearning": "RW"})
        df_rows_agg = df_rows.groupby(["algorithm", "dataset_percentage"])["run_time_seconds"].mean().reset_index()

        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.tick_params(axis="both", labelsize=11)
        ax.xaxis.label.set_size(13)
        ax.xaxis.label.set_weight("bold")
        ax.yaxis.label.set_size(13)
        ax.yaxis.label.set_weight("bold")
        for i, algo in enumerate(df_rows_agg["algorithm"].unique()):
            sub = df_rows_agg[df_rows_agg["algorithm"] == algo].sort_values("dataset_percentage")
            color = WONG_LIST[i % len(WONG_LIST)]
            marker = ALGO_MARKER.get(display_algo(algo), "o")
            ax.plot(sub["dataset_percentage"], sub["run_time_seconds"], marker=marker, linewidth=2.5,
                   color=color, label=algo, markersize=8)
        ax.set_xlabel("Dataset Percentage")
        ax.set_ylabel("Avg Runtime (s)")
        ax.set_xticks(ROW_PERCENTAGES)
        ax.set_xticklabels([f"{int(x * 100)}%" for x in ROW_PERCENTAGES])
        ax.legend(loc="best", fontsize=11, frameon=True, title=None)
        ax.grid(True, alpha=0.3)
        if ds == "so":
            ax.set_yscale("log")
        save_pdf(fig, f"{ds}_scalability_rows_graph.pdf",
                 f"Scalability ({ds.upper()}): Average runtime in seconds for BruteForce and RW versus dataset percentage (10% to 100% of rows).")
    else:
        print(f"  ⚠ Skipping {ds} rows: {row_file} not found")

print("Done!\n")

## Benchmark Figures — Problem 2 & 3 (3 PDFs)

**Data source:** Problem 2: **`problem_2_3_algorithms/benchmark_find_delta.py`** → `find_delta_benchmark_results.csv` (and .xlsx). Problem 3: **`problem_2_3_algorithms/benchmark_find_epsilon.py`** → `find_epsilon_benchmark_results.csv` (and .xlsx). Results live under `benchmark_two_phase_FIXED` (SO) and `benchmark_acs_full` (ACS). ACS Problem 3 excludes Rule 10 due to outlier runtimes.

**What each figure shows:**
1. **benchmark_problem2_runtime_vs_epsilon.pdf** — Problem 2 (largest δ): average runtime vs ε for SO and ACS rules.
2. **benchmark_problem3_runtime_vs_delta_SO_only.pdf** — Problem 3 (smallest ε): average runtime vs δ (% of dataset), SO dataset only.
3. **benchmark_problem3_runtime_vs_delta_ACS_only.pdf** — Problem 3 (smallest ε): average runtime vs δ (% of dataset), ACS dataset only.


In [5]:
print("📊 Generating benchmark figures...")

import ast as _ast

def _full_rule_label(cond_str, treat_str):
    """Return (condition_text, treatment_text) with full readable strings."""
    try:
        c = _ast.literal_eval(cond_str)
        t = _ast.literal_eval(treat_str)
    except Exception:
        return cond_str, treat_str
    return (", ".join(f"{k} = {v}" for k, v in c.items()),
            ", ".join(f"{k} = {v}" for k, v in t.items()))


def _make_benchmark_figure(df_so, df_acs, x_col, x_label, pdf_name, title=None, use_delta_pct_for_acs=False, convert_to_k=False):
    """Create a benchmark chart with averaged lines for SO and ACS datasets.
    Each dataset shows average runtime across all rules with error bars (std dev).
    
    Args:
        use_delta_pct_for_acs: If True, for ACS dataset, group by delta percentage 
                               instead of absolute delta (for Problem 3).
        convert_to_k: If True, divide x-axis values by 1000 (for epsilon in Problem 2).
    """

    fig, ax = plt.subplots(figsize=(8, 6))

    # Process SO dataset
    if df_so is not None and len(df_so) > 0:
        so_grouped = df_so.groupby(x_col)["Runtime_Seconds"].agg(['mean', 'std']).reset_index()
        so_grouped = so_grouped.sort_values(x_col)
        so_grouped['std'] = so_grouped['std'].fillna(0)  # Handle NaN when only one data point
        x_vals_so = so_grouped[x_col] / 1000 if convert_to_k else so_grouped[x_col]
        ax.plot(x_vals_so, so_grouped['mean'],
                marker="o", linewidth=2.5, color=WONG["blue"], 
                label="SO (average)", markersize=8)
        ax.errorbar(x_vals_so, so_grouped['mean'], 
                   yerr=so_grouped['std'], 
                   color=WONG["blue"], alpha=0.3, capsize=3, capthick=1)

    # Process ACS dataset
    if df_acs is not None and len(df_acs) > 0:
        # Exclude Rule 10 which has outliers (998s and 968s runtime) - likely a benchmark error
        df_acs = df_acs[df_acs['Rule_ID'] != 10].copy()
        
        # For Problem 3, use delta percentage for ACS instead of absolute delta
        if use_delta_pct_for_acs and x_col == "Delta":
            # Calculate delta percentage
            df_acs['Delta_Percentage'] = (df_acs['Delta'] / df_acs['Dataset_Size'] * 100).round(2)
            group_col = 'Delta_Percentage'
            x_label_acs = "δ (% of dataset size)"
            x_vals_acs = None  # Not applicable for percentage
        else:
            group_col = x_col
            x_label_acs = x_label
            x_vals_acs = None  # Will be set after grouping
        
        acs_grouped = df_acs.groupby(group_col)["Runtime_Seconds"].agg(['mean', 'std']).reset_index()
        acs_grouped = acs_grouped.sort_values(group_col)
        acs_grouped['std'] = acs_grouped['std'].fillna(0)  # Handle NaN when only one data point
        if x_vals_acs is None:
            x_vals_acs = acs_grouped[group_col] / 1000 if convert_to_k else acs_grouped[group_col]
        label_acs = "ACS (average)" if not use_delta_pct_for_acs else "ACS (average, excluding Rule 10)"
        ax.plot(x_vals_acs, acs_grouped['mean'],
                marker="s", linewidth=2.5, color=WONG["orange"], 
                label=label_acs, markersize=8)
        ax.errorbar(x_vals_acs, acs_grouped['mean'], 
                   yerr=acs_grouped['std'], 
                   color=WONG["orange"], alpha=0.3, capsize=3, capthick=1)

    ax.set_yscale("log")
    # Update x-axis label if converting to K
    if convert_to_k and "ε" in x_label:
        final_x_label = "ε (K, fixed for each experiment)"
    else:
        final_x_label = x_label
    ax.set_xlabel(final_x_label, fontweight="bold")
    ax.set_ylabel("Runtime (seconds, log scale)", fontweight="bold")
    ax.legend(loc="best", fontsize=11, frameon=True)
    ax.grid(True, alpha=0.3, which="both")

    save_pdf(fig, pdf_name, title=title)


# ── Problem 2: Runtime vs Epsilon ─────────────────────────────────────
print("📊 Generating Problem 2 figure (Runtime vs Epsilon)...")
p2_so = pd.read_csv(BENCH_P2_SO_CSV) if os.path.exists(BENCH_P2_SO_CSV) else None
p2_acs = pd.read_csv(BENCH_P2_ACS_CSV) if os.path.exists(BENCH_P2_ACS_CSV) else None
_make_benchmark_figure(
    p2_so, p2_acs, x_col="Epsilon",
    x_label="ε (fixed for each experiment)",
    pdf_name="benchmark_problem2_runtime_vs_epsilon.pdf",
    title="Benchmark Problem 2 (find largest delta): Average runtime versus epsilon for SO and ACS dataset rules.",
    convert_to_k=True)  # Convert epsilon values to thousands (K)

# ── Problem 3: Runtime vs Delta ───────────────────────────────────────
print("📊 Generating Problem 3 figures (Runtime vs Delta)...")
p3_so = pd.read_csv(BENCH_P3_SO_CSV) if os.path.exists(BENCH_P3_SO_CSV) else None
p3_acs = pd.read_csv(BENCH_P3_ACS_CSV) if os.path.exists(BENCH_P3_ACS_CSV) else None

# Version 1: SO only (absolute delta)
print("  Generating SO-only figure...")
if p3_so is not None and len(p3_so) > 0:
    so_grouped = p3_so.groupby("Delta")["Runtime_Seconds"].agg(['mean', 'std']).reset_index()
    so_grouped = so_grouped.sort_values("Delta")
    so_grouped['std'] = so_grouped['std'].fillna(0)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(so_grouped["Delta"], so_grouped['mean'],
            marker="o", linewidth=2.5, color=WONG["blue"], 
            label="SO (average)", markersize=8)
    ax.errorbar(so_grouped["Delta"], so_grouped['mean'], 
               yerr=so_grouped['std'], 
               color=WONG["blue"], alpha=0.3, capsize=3, capthick=1)
    ax.set_yscale("log")
    ax.set_xlabel("δ (minimum subgroup size)", fontweight="bold")
    ax.set_ylabel("Runtime (seconds, log scale)", fontweight="bold")
    ax.legend(loc="best", fontsize=11, frameon=True)
    ax.grid(True, alpha=0.3, which="both")
    save_pdf(fig, "benchmark_problem3_runtime_vs_delta_SO_only.pdf",
             "Benchmark Problem 3 (find smallest epsilon): Average runtime versus delta (% of dataset), SO dataset only.")

# Version 2: ACS only (delta percentage)
print("  Generating ACS-only figure...")
if p3_acs is not None and len(p3_acs) > 0:
    # Exclude Rule 10 which has outliers
    p3_acs_clean = p3_acs[p3_acs['Rule_ID'] != 10].copy()
    p3_acs_clean['Delta_Percentage'] = (p3_acs_clean['Delta'] / p3_acs_clean['Dataset_Size'] * 100).round(2)
    
    acs_grouped = p3_acs_clean.groupby('Delta_Percentage')["Runtime_Seconds"].agg(['mean', 'std']).reset_index()
    acs_grouped = acs_grouped.sort_values('Delta_Percentage')
    acs_grouped['std'] = acs_grouped['std'].fillna(0)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(acs_grouped['Delta_Percentage'], acs_grouped['mean'],
            marker="s", linewidth=2.5, color=WONG["orange"], 
            label="ACS (average, excluding Rule 10)", markersize=8)
    ax.errorbar(acs_grouped['Delta_Percentage'], acs_grouped['mean'], 
               yerr=acs_grouped['std'], 
               color=WONG["orange"], alpha=0.3, capsize=3, capthick=1)
    ax.set_yscale("log")
    ax.set_xlabel("δ (% of dataset size)", fontweight="bold")
    ax.set_ylabel("Runtime (seconds, log scale)", fontweight="bold")
    ax.legend(loc="best", fontsize=11, frameon=True)
    ax.grid(True, alpha=0.3, which="both")
    save_pdf(fig, "benchmark_problem3_runtime_vs_delta_ACS_only.pdf",
             "Benchmark Problem 3 (find smallest epsilon): Average runtime versus delta (% of dataset), ACS dataset only.")

print("✅ Done! Generated 2 separate versions of Problem 3 figure.\n")

📊 Generating benchmark figures...
📊 Generating Problem 2 figure (Runtime vs Epsilon)...
  ✅ Saved: benchmark_problem2_runtime_vs_epsilon.pdf
📊 Generating Problem 3 figures (Runtime vs Delta)...
  Generating SO-only figure...


  ✅ Saved: benchmark_problem3_runtime_vs_delta_SO_only.pdf
  Generating ACS-only figure...


  ✅ Saved: benchmark_problem3_runtime_vs_delta_ACS_only.pdf
✅ Done! Generated 2 separate versions of Problem 3 figure.



## Summary


In [ ]:
# List all generated PDFs
pdfs = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".pdf")])
print(f"🎉 Total PDF figures generated: {len(pdfs)}\n")
for f in pdfs:
    print(f"  📄 {f}")


🎉 Total PDF figures generated: 15

  📄 ablation_acs_correctness_varying_delta.pdf
  📄 ablation_acs_correctness_varying_epsilon.pdf
  📄 ablation_acs_homogeneity_impact_delta.pdf
  📄 ablation_acs_homogeneity_impact_epsilon.pdf
  📄 ablation_acs_runtime_varying_delta.pdf
  📄 ablation_acs_runtime_varying_epsilon.pdf
  📄 ablation_so_correctness_varying_delta.pdf
  📄 ablation_so_correctness_varying_epsilon.pdf
  📄 ablation_so_homogeneity_impact_delta.pdf
  📄 ablation_so_homogeneity_impact_epsilon.pdf
  📄 ablation_so_runtime_varying_delta.pdf
  📄 ablation_so_runtime_varying_epsilon.pdf
  📄 benchmark_problem2_runtime_vs_epsilon.pdf
  📄 benchmark_problem3_runtime_vs_delta_ACS_only.pdf
  📄 benchmark_problem3_runtime_vs_delta_SO_only.pdf
